In [1]:
import tensorflow as tf 
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from tensorflow.keras.utils import to_categorical
import joblib

In [2]:
# loading and preparing dataset
data = pd.read_csv("StressLevelDataset.csv")

X = data.iloc[:, :-1] # selects all the columns except the last one as features
Y = data.iloc[:, -1] # selects the last column as the target
X = pd.get_dummies(X)   # one hot encoding

y_encoded = to_categorical(Y, num_classes=3)

X_train, X_test, Y_train, Y_test = train_test_split(X, y_encoded, test_size = 0.3, random_state = 109) #splits the dataset into training and testing data

# standardizing data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA(n_components=2)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

joblib.dump(scaler, 'scaler.joblib')
joblib.dump(pca, 'pca.joblib')
np.save('X_test.npy', X_test)
np.save('Y_test.npy', Y_test)
np.save('y_original.npy', Y) 

In [3]:
model = models.Sequential()
model.add(layers.InputLayer(shape=(2,)))

model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.3))

model.add(layers.Dense(64, activation='relu'))

model.add(layers.Dense(32, activation='relu'))

model.add(layers.Dense(3, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,819 (42.26 KB)

 Trainable params: 10,819 (42.26 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# model.fit(X_train, Y_train, epochs=50, batch_size=32, validation_data=(X_test, Y_test))

# test_loss, test_acc = model.evaluate(X_test, Y_test)
# print(f"Accuracy: {test_acc}")

history = model.fit(X_train, Y_train, epochs=50, batch_size=32, validation_data=(X_test, Y_test))

test_loss, test_acc = model.evaluate(X_test, Y_test)
print(f"Accuracy: {test_acc}")

# Save the training history
joblib.dump(history.history, 'training_history.joblib')

y_pred = model.predict(X_test)
np.save('y_pred.npy', y_pred)

# Save the model
model.save('stress_level_model.h5')

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5274 - loss: 0.8836 - val_accuracy: 0.8848 - val_loss: 0.4303
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8585 - loss: 0.4347 - val_accuracy: 0.8818 - val_loss: 0.2503
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8998 - loss: 0.2310 - val_accuracy: 0.8909 - val_loss: 0.2255
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8647 - loss: 0.2531 - val_accuracy: 0.8909 - val_loss: 0.2199
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8730 - loss: 0.2246 - val_accuracy: 0.8879 - val_loss: 0.2158
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8897 - loss: 0.2163 - val_accuracy: 0.8879 - val_loss: 0.2134
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8877 - loss: 0.2154 - val_accuracy: 0.8818 - val_loss: 0.2136
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8816 - loss: 0.2122 - val_accuracy: 0.8818 - val_loss:

In [5]:
y_pred = model.predict(X_test)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
